# Some quick checks / scripts


### Check File Already Exists


In [ ]:
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

client = AzureOpenAI()

# List all uploaded files
files = client.files.list()

# Check for file by name
file_name_to_check = "0.pdf"
file_exists = any(f.filename == file_name_to_check for f in files)

print("File exists:", file_exists)


### Delete All Files


In [ ]:
import os
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI()

# Remove all files from the Azure OpenAI file storage
files = client.files.list()
for file in files:
    print(f"Deleting file: {file.filename} with ID: {file.id}")
    client.files.delete(file_id=file.id)

### Delete All Single Pages On Azure


In [ ]:
from openai import AzureOpenAI
from dotenv import load_dotenv

load_dotenv()

client = AzureOpenAI()

# Remove all files from the Azure OpenAI file storage
files = client.files.list()
for file in files:
    if len(file.filename.split("_")) > 1 and file.filename.split("_")[1] == "page":
        print(f"Deleting file: {file.filename} with ID: {file.id}")
        client.files.delete(file_id=file.id)

### Reset Metadata


In [ ]:
import json
from pathlib import Path


metadata_file = Path.cwd().parent / "data/extracted_data/metadata.json"
metadata = json.loads(metadata_file.open("rb").read())

for metadata_item in metadata.values():
    print(f"Metadata item: {metadata_item}")
    metadata_item.pop("metadata")

# Save the modified metadata back to the file
with metadata_file.open("w") as f:
    json.dump(metadata, f, indent=2)


### Reset Metadata Pages


In [ ]:
import json
from pathlib import Path


metadata_file = Path.cwd().parent / "data/extracted_data/metadata.json"
metadata = json.loads(metadata_file.open("rb").read())

for metadata_item in metadata.values():
    print(f"Metadata item: {metadata_item}")
    if "pages" in metadata_item:
        metadata_item.pop("pages")

# Save the modified metadata back to the file
with metadata_file.open("w") as f:
    json.dump(metadata, f, indent=2)


### Reset Questions References


In [ ]:
import json
from pathlib import Path


questions_file = Path.cwd().parent / "data/generated_data/generated_questions.json"
questions_info = json.loads(questions_file.open("rb").read())

for question_type, questions in questions_info.items():
    for question_id, question_item in questions.items():
        if len(question_item["files"]) > 2:
            questions_info[question_type][question_id]["files"] = questions_info[
                question_type
            ][question_id]["files"][:2]

# Save the modified metadata back to the file
with questions_file.open("w") as f:
    json.dump(questions_info, f, indent=2)


### Tags Analysis


In [ ]:
from pathlib import Path
import json

# Grab the metadata file
metadata_file = Path.cwd().parent / "data/extracted_data/metadata.json"
metadata = json.loads(metadata_file.open("rb").read())
tags = [
    doc["metadata"]["tags"]
    for doc in metadata.values()
    if "metadata" in doc and "tags" in doc["metadata"]
]

In [ ]:
# Distribution of tags
from collections import Counter
import matplotlib.pyplot as plt

# Flatten the list of tag lists and normalize to lowercase
flat_tags = [tag.lower() for tag_list in tags for tag in tag_list]
tag_counts = Counter(flat_tags)

# Filter tags with more than 10 counts
filtered_tag_counts = {tag: count for tag, count in tag_counts.items() if count > 15}

# Sort filtered tags by count (descending)
sorted_tags = sorted(filtered_tag_counts.items(), key=lambda x: x[1], reverse=True)
tags_sorted, counts_sorted = zip(*sorted_tags) if sorted_tags else ([], [])

plt.figure(figsize=(12, 6))
plt.bar(tags_sorted, counts_sorted, color="orange")
plt.xlabel("Tags")
plt.ylabel("Count")
plt.title("Tags with More Than 10 Occurrences (Sorted)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Pages Tags Analysis


In [ ]:
from pathlib import Path
import json

# Grab the metadata file
metadata_file = Path.cwd().parent / "data/extracted_data/metadata.json"
metadata = json.loads(metadata_file.open("rb").read())
tags = []
for corpus_metadata in metadata.values():
    if "pages" in corpus_metadata:
        # Extend the tags list with tags from each page
        tags.extend(
            [page["tags"] for page in corpus_metadata["pages"] if "tags" in page]
        )

In [ ]:
# Distribution of tags
from collections import Counter
import matplotlib.pyplot as plt

# Flatten the list of tag lists and normalize to lowercase
flat_tags = [tag.lower() for tag_list in tags for tag in tag_list]
tag_counts = Counter(flat_tags)

# Filter tags with more than 10 counts
filtered_tag_counts = {tag: count for tag, count in tag_counts.items() if count > 15}

# Sort filtered tags by count (descending)
sorted_tags = sorted(filtered_tag_counts.items(), key=lambda x: x[1], reverse=True)
tags_sorted, counts_sorted = zip(*sorted_tags) if sorted_tags else ([], [])

plt.figure(figsize=(12, 6))
plt.bar(tags_sorted, counts_sorted, color="orange")
plt.xlabel("Tags")
plt.ylabel("Count")
plt.title("Tags with More Than 10 Occurrences (Sorted)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Sanitize Metadata


In [ ]:
from pathlib import Path
import json
from extractor import create_metadata

from openai import AzureOpenAI
from langfuse import Langfuse
from dotenv import load_dotenv

load_dotenv()

openai_client = AzureOpenAI()
langfuse_client = Langfuse()
model_name = "gpt-4o-v2"
data_folder = Path.cwd().parent / "data"

current_metadata = json.loads(
    (data_folder / "extracted_data/metadata.json").open("rb").read()
).values()


def check_metadata_extraction(metadata):
    """Check if metadata extraction is complete."""
    return all(
        "metadata" in item
        and "topic" in item["metadata"]
        and "summary" in item["metadata"]
        and item["metadata"]["topic"] != ""
        and item["metadata"]["summary"] != ""
        for item in metadata
    )


if not check_metadata_extraction(current_metadata):
    print("Some metadata hasn't been extracted, retrying...")
    create_metadata(langfuse_client, openai_client, data_folder, model_name)

if not check_metadata_extraction(current_metadata):
    print(
        "Some metadata hasn't been extracted, please check the documents and try again."
    )

### Check Page Metadata


In [ ]:
import json
from pathlib import Path


metadata_file = Path.cwd().parent / "data/extracted_data/metadata.json"
metadata = json.loads(metadata_file.open("rb").read())

count = 0

for metadata_item in metadata.values():
    if "pages" in metadata_item:
        for page in metadata_item["pages"]:
            if "topic" in page or "summary" in page:
                count += 1 if page["topic"] == "" or page["summary"] == "" else 0

print(f"Total pages with missing topic or summary: {count}")

### Delete Test Run


In [ ]:
from openai import AzureOpenAI
from dotenv import load_dotenv
from pathlib import Path
import json

load_dotenv()

client = AzureOpenAI()

# Load metadata
metadata = {}
metadata_file = Path.cwd().parent / "data_2/extracted_data/metadata.json"
if metadata_file.exists():
    metadata = json.loads(metadata_file.open("rb").read())
else:
    raise FileNotFoundError(f"Metadata file not found: {metadata_file}")

# Remove all test files from the Azure OpenAI file storage
test_files = []
for metadata_item in metadata.values():
    test_files.append(metadata_item["openai_file_id"])
    if "pages" in metadata_item:
        for page_info in metadata_item["pages"].values():
            test_files.append(page_info["openai_file_id"])

for file_id in test_files:
    print(f"Deleting file with ID: {file_id}")
    client.files.delete(file_id=file_id)